# Session 4: Multi-Platform Market Comparison

In this notebook we will do Market Analysis + Multi-Platform Comparisons.

We will:
- Load cleaned Movies + TV datasets
- Standardize fields and build a unified analysis table
- Create **pivot tables** for market comparisons
- Visualize with **stacked bar**, **grouped bar**, and **heatmap**
- Write **data-backed strategic insights**


## 0) Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Make plots larger and readable in class
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path("G:\The US\Gradute School\Build Project\Streaming wars\streaming-bi-template\streaming-bi-template\Data")  # adjust if needed

MOVIES_PATH = DATA_DIR / "MoviesOnStreamingPlatforms_Cleaned.csv"
TV_PATH     = DATA_DIR / "TVShowsOnStreamingPlatforms_Cleaned.csv"

print('Movies file:', MOVIES_PATH)
print('TV file:    ', TV_PATH)


<>:9: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<>:9: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
C:\Users\gideo\AppData\Local\Temp\ipykernel_23780\754552419.py:9: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
  DATA_DIR = Path("G:\The US\Gradute School\Build Project\Streaming wars\streaming-bi-template\streaming-bi-template\Data")  # adjust if needed


## 1) Load Data

In [ ]:
movies = pd.read_csv(MOVIES_PATH)
tv     = pd.read_csv(TV_PATH)

print("Movies shape:", movies.shape)
print("TV shape:    ", tv.shape)

display(movies.head())


## 2) Quick Data Audit (Columns + Missingness)
We want to confirm the key analysis columns exist:
- `platform` (or `Netflix`, `Hulu`, `Prime Video`, `Disney+` style indicator)
- `year` / release year
- `genre` (or a field we can map)

Because cleaned datasets can differ, we’ll **detect schema** and standardize.


In [ ]:
def summarize_df(df, name):
    print(f"\n==== {name} ====")
    print("Columns:", list(df.columns))
    print("\nMissingness (top 15):")
    display(df.isna().mean().sort_values(ascending=False).head(15))
    print("\nDtypes (top 15):")
    display(df.dtypes.head(15))

summarize_df(movies, "Movies")
summarize_df(tv, "TV Shows")


## 3) Standardize Schema
We will build a unified table with these columns:
- `title`
- `type` (Movie / TV)
- `platform` (one row per title per platform)
- `year` (release year)
- `genre` (primary genre or a normalized genre label)

### Platform formats supported
1) A single text column like `platform`
2) Four binary columns like `Netflix`, `Hulu`, `Prime Video`, `Disney+`


In [ ]:
# --- Helpers ---

PLATFORMS = [
    "Netflix", "Hulu", "Prime Video", "Disney+"
]

def melt_binary_platforms(df, platform_cols):
    # Expect platform columns are 0/1 or True/False
    tmp = df.copy()
    for c in platform_cols:
        tmp[c] = tmp[c].fillna(0)
    long = tmp.melt(
        id_vars=[c for c in tmp.columns if c not in platform_cols],
        value_vars=platform_cols,
        var_name="platform",
        value_name="on_platform"
    )
    long = long[long["on_platform"].astype(int) == 1].drop(columns=["on_platform"])
    return long

def standardize_common_fields(df):
    out = df.copy()

    # title
    if "title" not in out.columns:
        for cand in ["Title", "name", "Name"]:
            if cand in out.columns:
                out = out.rename(columns={cand: "title"})
                break

    # year
    if "year" not in out.columns:
        for cand in ["Year", "release_year", "Release Year", "released", "Release_Year"]:
            if cand in out.columns:
                out = out.rename(columns={cand: "year"})
                break

    # genre
    if "genre" not in out.columns:
        for cand in ["Genre", "genres", "listed_in", "Listed In", "category"]:
            if cand in out.columns:
                out = out.rename(columns={cand: "genre"})
                break

    # clean year
    if "year" in out.columns:
        out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")

    # clean genre: keep only the first genre if comma-separated
    if "genre" in out.columns:
        out["genre"] = out["genre"].astype(str).str.strip()
        out["genre"] = out["genre"].replace({"nan": np.nan})
        out["genre_primary"] = out["genre"].str.split(",").str[0].str.strip()
    else:
        out["genre_primary"] = np.nan

    # Rotten Tomatoes Score
    if "RottenTomatoes_Score" in out.columns:
        out["rotten_tomatoes_score"] = out["RottenTomatoes_Score"]

    return out

def standardize_to_long(df, content_type):
    df = standardize_common_fields(df)

    out = melt_binary_platforms(df, PLATFORMS)
    
    out["platform"] = out["platform"].astype(str).str.strip()

    out["type"] = content_type

    # Minimal required columns
    keep = []
    for c in ["title", "type", "platform", "year", "genre_primary", "rotten_tomatoes_score"]:
        if c in out.columns:
            keep.append(c)
    out = out[keep].copy()

    # Standardize column names
    if "genre_primary" in out.columns:
        out = out.rename(columns={"genre_primary": "genre"})
    else:
        out["genre"] = np.nan

    return out

movies_long = standardize_to_long(movies, "Movie")
tv_long     = standardize_to_long(tv, "TV Show")

df = pd.concat([movies_long, tv_long], ignore_index=True)

print("Unified df shape:", df.shape)
display(df.head())


## 4) Basic Quality Checks
We’ll check for:
- Missing `platform` / `year` / `genre`
- Duplicate title-platform-type rows


In [ ]:
# Missingness
display(df.isna().mean().sort_values(ascending=False))

# Duplicates (title + platform + type)
dup_mask = df.duplicated(subset=["title", "platform", "type"], keep=False)
dups = df[dup_mask].sort_values(["title", "platform", "type"])
print("Duplicate rows:", dups.shape[0])
display(dups.head(10))


## 5) Pivot Tables for Market Comparison
These pivots are the *foundation* for all charts.

### A) Total content count by platform


In [ ]:
pivot_total = df.pivot_table(
    index="platform",
    values="title",
    aggfunc="count"
).rename(columns={"title": "title_count"}).sort_values("title_count", ascending=False)

display(pivot_total)


### B) Content count by platform and type (Movie vs TV)

In [ ]:
pivot_type = df.pivot_table(
    index="platform",
    columns="type",
    values="title",
    aggfunc="count",
    fill_value=0
).sort_values(by=list(df["type"].dropna().unique()), ascending=False)

display(pivot_type)


### C) Genre mix by platform (Top genres)
We’ll keep top genres to make charts readable.


In [ ]:
# Determine top N genres globally
TOP_N = 10
top_genres = (df["genre"]
              .dropna()
              .value_counts()
              .head(TOP_N)
              .index.tolist())

df_topg = df[df["genre"].isin(top_genres)].copy()

pivot_genre = df_topg.pivot_table(
    index="platform",
    columns="genre",
    values="title",
    aggfunc="count",
    fill_value=0
)

display(pivot_genre)


## 6) Charts
We will create:
- Stacked bar: **genre composition** by platform
- Grouped bar: **volume comparison** by platform and type
- Heatmap: **release year patterns** across platforms


### 6.1 Stacked Bar — Genre Composition (Top Genres)

In [ ]:
# Stacked bar chart from pivot table
ax = pivot_genre.loc[pivot_total.index].plot(kind="bar", stacked=True)
ax.set_title("Genre Mix by Platform (Top Genres)")
ax.set_xlabel("Platform")
ax.set_ylabel("Number of Titles")
plt.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Teaching prompt:
# - Which platform looks most diversified?
# - Which genres dominate for each platform?


### 6.2 Grouped Bar — Volume Comparison (Movies vs TV Shows)

In [ ]:
ax = pivot_type.loc[pivot_total.index].plot(kind="bar")
ax.set_title("Content Volume by Platform and Type")
ax.set_xlabel("Platform")
ax.set_ylabel("Number of Titles")
plt.legend(title="Type")
plt.tight_layout()
plt.show()

# Teaching prompt:
# - Which platform is strongest in movies vs TV?
# - Does a platform look 'balanced' or specialized?


### 6.3 Heatmap — Release Year Patterns by Platform
We’ll bin years into ranges to reduce noise.


In [ ]:
# Filter reasonable year range (optional; adjust if your data is older/newer)
df_year = df[df["year"].between(1950, 2030)].copy()

# Create year bins (e.g., 5-year buckets)
BIN_SIZE = 5
df_year["year_bin"] = (df_year["year"] // BIN_SIZE) * BIN_SIZE
df_year["year_bin"] = df_year["year_bin"].astype("Int64")

pivot_year = df_year.pivot_table(
    index="platform",
    columns="year_bin",
    values="title",
    aggfunc="count",
    fill_value=0
).loc[pivot_total.index]

# Plot heatmap using matplotlib (no seaborn)
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot_year.values, aspect="auto")

ax.set_title(f"Release Year Heatmap (Binned by {BIN_SIZE} Years)")
ax.set_yticks(range(len(pivot_year.index)))
ax.set_yticklabels(pivot_year.index)

ax.set_xticks(range(len(pivot_year.columns)))
ax.set_xticklabels(pivot_year.columns, rotation=90)

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Number of Titles")

plt.tight_layout()
plt.show()

display(pivot_year.head())


### 6.4 Box Plot — Score Distribution by Platform

In [ ]:
df_score = df.copy()
df_score["rotten_tomatoes_score"] = pd.to_numeric(df_score["rotten_tomatoes_score"], errors="coerce")

# Keep only rows with valid platform + score
df_score = df_score.dropna(subset=["platform", "rotten_tomatoes_score"])

platform_order = (
    df_score.groupby("platform")["rotten_tomatoes_score"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

data = [df_score.loc[df_score["platform"] == p, "rotten_tomatoes_score"].values for p in platform_order]

plt.figure(figsize=(12, 6))
plt.boxplot(data, labels=platform_order, showfliers=True)
plt.title("RottenTomatoes Score Distribution by Platform")
plt.xlabel("Platform")
plt.ylabel("RottenTomatoes Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 6.5 Mean score by platform and type

In [ ]:
summary_type = (
    df_score.groupby(["platform", "type"])["rotten_tomatoes_score"]
    .agg(["count", "mean"])
    .reset_index()
)

# Pivot for grouped bar chart
pivot_score = summary_type.pivot(index="platform", columns="type", values="mean").fillna(0)

# Order platforms by overall mean
platform_order = (
    df_score.groupby("platform")["rotten_tomatoes_score"]
    .mean()
    .sort_values(ascending=False)
    .index
)
pivot_score = pivot_score.loc[platform_order]

ax = pivot_score.plot(kind="bar")
ax.set_title("Average RottenTomatoes Score by Platform and Type")
ax.set_xlabel("Platform")
ax.set_ylabel("Average RottenTomatoes Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

display(pivot_score)

## 7) Interpretation: Turning Charts into Market Positioning
## Strategic Recommendations
1) Continue leveraging massive content volume as a primary competitive moat, ensuring users never feel the need to look elsewhere for sheer variety
2) Implement advanced, personalized recommendation engines to cure user "choice paralysis" and surface hidden gems from the massive catalog
3) Shift a portion of the budget toward high-quality, episodic original series to drive weekly cultural conversations and improve average critical reception
4) Expand investment in unscripted and reality television, which the data shows is a top-performing, highly engaging, and cost-effective genre
5) Deeper integration of Prime Video with the broader Amazon retail ecosystem (e.g., shopping tie-ins, exclusive Prime perks) to maximize overall subscriber lifetime value
## Expected Impact:
1) Balancing the massive movie library with more episodic TV shows will reduce churn and keep users subscribed month-to-month
2) Improving discoverability and UI curation will reduce scrolling time and increase the average daily watch-time per user
3) Investing in prestige originals will shift Prime Video's reputation from just a "catch-all volume library" to a premium, award-winning entertainment destination
4) Continuing to scale youth content alongside its massive 18+ library will solidify its position as the ultimate household-shared platform
5) Higher engagement on Prime Video directly correlates to higher retention for the overarching Amazon Prime subscription, driving broader retail and ecosystem success

In [ ]:
# Quick helper to compute platform shares within top genres (optional for discussion)
genre_share = pivot_genre.div(pivot_genre.sum(axis=1), axis=0).fillna(0)
display((genre_share * 100).round(1))
